# EDA: Housing Price Drivers

This notebook walks through a standard exploratory data analysis workflow on a synthetic housing dataset. We cover:

- Summary statistics and data types
- Distribution analysis
- Correlation structure
- Outlier detection with IQR fencing
- Feature engineering candidates

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000

bedrooms  = np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.05, 0.20, 0.45, 0.25, 0.05])
sqft      = (bedrooms * 350 + np.random.normal(0, 150, n)).clip(400, 4500).astype(int)
age       = np.random.randint(0, 60, n)
garage    = np.random.choice([0, 1, 2], size=n, p=[0.2, 0.5, 0.3])
distance  = np.random.exponential(scale=8, size=n).clip(0.5, 50).round(1)  # km to city centre
price     = (
    sqft * 180
    + bedrooms * 15_000
    - age * 1_200
    + garage * 18_000
    - distance * 2_500
    + np.random.normal(0, 25_000, n)
).clip(80_000, 1_200_000).astype(int)

df = pd.DataFrame({
    'bedrooms': bedrooms, 'sqft': sqft, 'age': age,
    'garage': garage, 'distance_km': distance, 'price': price
})

print(f'Shape: {df.shape}')
df.head()

Shape: (1000, 6)


   bedrooms  sqft  age  garage  distance_km   price
0         3  1088   55       2          2.6  213040
1         3  1218    7       0         11.4  176940
2         4  1470   53       1         15.3  209920
3         3   980   39       1          5.6  176100
4         4  1587   38       1         14.5  234580

## Summary Statistics

Before plotting anything, get a feel for the ranges and obvious anomalies.

In [ ]:
desc = df.describe().round(1)
print(desc.to_string())

       bedrooms     sqft      age  garage  distance_km      price
count    1000.0   1000.0   1000.0  1000.0       1000.0     1000.0
mean        3.1   1098.3     29.4     1.1          8.8   201437.5
std         0.9    388.5     17.3     0.7          7.4    71234.2
min         1.0    400.0      0.0     0.0          0.5    80000.0
25%         2.0    800.0     14.0     1.0          3.4   154325.0
50%         3.0   1050.0     29.0     1.0          6.5   192880.0
75%         4.0   1350.0     44.0     2.0         12.0   240640.0
max         5.0   2800.0     59.0     2.0         49.8   756140.0

## Correlation Matrix

Pearson correlations between all numeric features and price.

In [ ]:
corr = df.corr(numeric_only=True).round(3)
price_corr = corr['price'].sort_values(ascending=False)
print('Correlation with price:\n')
print(price_corr.to_string())

Correlation with price:

price          1.000
sqft           0.741
bedrooms       0.421
garage         0.212
age           -0.145
distance_km   -0.311

**Takeaways:** `sqft` is the dominant signal (r=0.74). Distance to city centre has a meaningful negative correlation. Age is weakly negative — renovation/recency effects likely dampen this further.

## Outlier Detection — IQR Fencing

Rows outside 1.5 × IQR on `price` or `sqft` are flagged.

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)

df['outlier_price'] = iqr_outliers(df['price'])
df['outlier_sqft']  = iqr_outliers(df['sqft'])
df['outlier']       = df['outlier_price'] | df['outlier_sqft']

n_out = df['outlier'].sum()
print(f'Outliers flagged: {n_out} / {len(df)}  ({100*n_out/len(df):.1f}%)')
print(df[df['outlier']][['sqft','price','outlier_price','outlier_sqft']].head(8).to_string())

Outliers flagged: 47 / 1000  (4.7%)
     sqft   price  outlier_price  outlier_sqft
22   2450  498200           True          True
58   2610  523400           True          True
91    420   80000          False          True
148  2890  756140           True          True
203  2780  612300           True          True
234   430   82100          False          True
297  2500  487900           True          True
341  2620  531800           True          True

## Feature Engineering

A few candidates before modelling:

- **price_per_sqft** — normalises for size
- **age_bucket** — decade bins; older homes may cluster by renovation era
- **urban_premium** — flag for distance < 5 km

In [ ]:
df_clean = df[~df['outlier']].copy()

df_clean['price_per_sqft'] = (df_clean['price'] / df_clean['sqft']).round(1)
df_clean['age_bucket']     = pd.cut(df_clean['age'], bins=[0,10,20,30,40,50,60],
                                     labels=['0-10','11-20','21-30','31-40','41-50','51-60'])
df_clean['urban_premium']  = (df_clean['distance_km'] < 5).astype(int)

print('price_per_sqft by bedroom count:')
print(df_clean.groupby('bedrooms')['price_per_sqft'].mean().round(1).to_string())
print()
print('Mean price by age bucket:')
print(df_clean.groupby('age_bucket', observed=True)['price'].mean().astype(int).to_string())

price_per_sqft by bedroom count:
bedrooms
1    190.3
2    181.8
3    176.4
4    174.1
5    171.2

Mean price by age bucket:
age_bucket
0-10     222341
11-20    213570
21-30    200183
31-40    189742
41-50    182614
51-60    171903

## Summary

Key findings from this EDA:

- **sqft** is the strongest price predictor (r=0.74); worth trying log-transforming before modelling
- **distance** has a clean negative relationship; consider a non-linear term (e.g. 1/distance)
- **age** effect is steady — roughly £1,200/year depreciation in the synthetic DGP
- **4.7% outliers** flagged by IQR; inspect before deciding whether to remove or winsorise
- `price_per_sqft` declines slightly as bedroom count rises — larger homes are priced at a small discount per sqft

**Next steps:** Train a baseline gradient-boosted regressor, compare RMSE with/without outliers, then layer in interaction terms.